# Rendimiento de cultivos del Valle del Cauca por temperatura

**Asignatura:** Ciencia de Datos con Python (IGS38) · Universidad Santiago de Cali

**Caso de estudio:** Analítica de datos abiertos de Colombia con datos.gov.co

**Parte 1:** Identificación de fuentes de datos y manipulación con Pandas

**Autor:** Juan David Muñoz Sotelo

## Pregunta
¿Cómo ha evolucionado el rendimiento de los principales cultivos del Valle del Cauca por subregión y qué tanto se asocia con la temperatura.

## Hipótesis
Los años con temperaturas más altas reducen el rendimiento (toneladas por hectárea) de los cultivos del Valle del Cauca, y este efecto es mayor en los cultivos transitorios que en los permanentes.

## Fuentes de datos

Todos los conjuntos se consultaron en [datos.gov.co](https://www.datos.gov.co) 

| Conjunto | ID | Rol en el análisis | Enlace |
|---|---|---|---|
| Producción de raíces, tubérculos y bulbos | `5m3h-gjmu` | Producción (transitorios) | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Producci-n-de-ra-ces-tub-rculos-y-bulbos-en-el-Val/5m3h-gjmu/data_preview) |
| Rendimiento del área cosechada, transitorios | `xb5v-aufn` | Variable objetivo (t/ha) | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Rendimiento-del-rea-cosechada-de-los-cultivos-tran/xb5v-aufn/data_preview) |
| Superficie sembrada, transitorios | `vs5v-e66i` | Contexto / área | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Superficie-sembrada-de-cultivos-transitorios-en-el/vs5v-e66i/data_preview) |
| Superficie sembrada, permanentes | `v4ub-9eme` | Contexto / área | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Superficie-sembrada-de-cultivos-permanentes-en-el-/v4ub-9eme/data_preview) |
| Consolidado agrícola, permanentes | `exfh-span` | Producción y rendimiento (permanentes) | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Consolidado-agr-cola-por-municipios-de-los-cultivo/exfh-span/data_preview) |
| Producción de hortalizas | `6bym-ygip` | Producción (transitorios) | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Producci-n-de-hortalizas-en-el-Valle-del-Cauca/6bym-ygip/data_preview) |
| Producción de frutales | `8xap-gecf` | Producción (mixto) | [Ver](https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Producci-n-de-frutales-en-el-Valle-del-Cauca/8xap-gecf/data_preview) |
| Temperatura ambiente del aire (IDEAM) | `sbwg-7ju4` | Variable explicativa | [Ver](https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Temperatura-Ambiente-del-Aire/sbwg-7ju4/data_preview) |

## Constantes y configuración

Se definen una sola vez los IDs de los conjuntos, la URL base de la API y el límite de filas de las muestras exploratorias.

In [15]:
import os
import pandas as pd
import time

os.makedirs("data", exist_ok=True)

BASE_URL = "https://www.datos.gov.co/resource"
LIMITE_COMPLETO = 1_000_000
LIMITE_TEMPERATURA = 1_000_000

LIMITE_TEMPERATURA = 10 
WHERE_DEPTO = "upper(departamento) LIKE '%VALLE DEL CAUCA%'"
T_MIN, T_MAX = -5, 45
ANIO_INI, ANIO_FIN = 2000, 2023  # el rango agrícola es 2000-2023
RUTA_TEMP_MENSUAL = "data/temperatura_valle.csv"
CARPETA_ANUAL = "data/temp_anual"

ID_RAICES = "5m3h-gjmu"
ID_REND_TRANS = "xb5v-aufn"
ID_SUP_TRANS = "vs5v-e66i"
ID_SUP_PERM = "v4ub-9eme"
ID_CONSOL_PERM = "exfh-span"
ID_HORTALIZAS = "6bym-ygip"
ID_FRUTALES = "8xap-gecf"
ID_TEMPERATURA = "sbwg-7ju4"

## Exploración inicial de las fuentes

Se guardan como CSV en `data/` (con sufijo `_muestra`) y se lee desde ese archivo para identificar sus columnas. Se lee todo como texto (`dtype=str`) para ver los datos tal como vienen; la conversión de tipos se hace después, en la limpieza.

### 1. Producción de raíces, tubérculos y bulbos (`5m3h-gjmu`)

In [3]:
URL = f"{BASE_URL}/{ID_RAICES}.csv?$limit={LIMITE_COMPLETO}"
RUTA_RAICES = "data/raices.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_RAICES, index=False)

raices = pd.read_csv(RUTA_RAICES, dtype=str)

print("Forma:", raices.shape)
print("\nColumnas:")
print(raices.columns.tolist())

Forma: (1773, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'produccion_toneladas']


### 2. Rendimiento del área cosechada, transitorios (`xb5v-aufn`)

In [4]:
URL = f"{BASE_URL}/{ID_REND_TRANS}.csv?$limit={LIMITE_COMPLETO}"
RUTA_REND_TRANS = "data/rend_trans.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_REND_TRANS, index=False)

rend_trans = pd.read_csv(RUTA_REND_TRANS, dtype=str)

print("Forma:", rend_trans.shape)
print("\nColumnas:")
print(rend_trans.columns.tolist())

Forma: (16748, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'rendimiento_toneladas_hectareas']


### 3. Superficie sembrada, transitorios (`vs5v-e66i`)

In [5]:
URL = f"{BASE_URL}/{ID_SUP_TRANS}.csv?$limit={LIMITE_COMPLETO}"
RUTA_SUP_TRANS = "data/sup_trans.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_SUP_TRANS, index=False)

sup_trans = pd.read_csv(RUTA_SUP_TRANS, dtype=str)

print("Forma:", sup_trans.shape)
print("\nColumnas:")
print(sup_trans.columns.tolist())

Forma: (16748, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'hectareas_sembradas']


### 4. Superficie sembrada, permanentes (`v4ub-9eme`)

In [6]:
URL = f"{BASE_URL}/{ID_SUP_PERM}.csv?$limit={LIMITE_COMPLETO}"
RUTA_SUP_PERM = "data/sup_perm.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_SUP_PERM, index=False)

sup_perm = pd.read_csv(RUTA_SUP_PERM, dtype=str)

print("Forma:", sup_perm.shape)
print("\nColumnas:")
print(sup_perm.columns.tolist())

Forma: (14590, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'hectareas_sembradas']


### 5. Consolidado agrícola, permanentes (`exfh-span`)

In [7]:
URL = f"{BASE_URL}/{ID_CONSOL_PERM}.csv?$limit={LIMITE_COMPLETO}"
RUTA_CONSOL_PERM = "data/consol_perm.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_CONSOL_PERM, index=False)

consol_perm = pd.read_csv(RUTA_CONSOL_PERM, dtype=str)

print("Forma:", consol_perm.shape)
print("\nColumnas:")
print(consol_perm.columns.tolist())

Forma: (15282, 11)

Columnas:
['tipo_cultivo', 'a_o', 'id_municipio', 'municipio', 'id_cultivo', 'cultivo', 'ciclo', 'hectareas_sembradas', 'hectareas_cosechadas', 'produccion_toneladas', 'rendimiento_toneladas']


### 6. Producción de hortalizas (`6bym-ygip`)

In [8]:
URL = f"{BASE_URL}/{ID_HORTALIZAS}.csv?$limit={LIMITE_COMPLETO}"
RUTA_HORTALIZAS = "data/hortalizas.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_HORTALIZAS, index=False)

hortalizas = pd.read_csv(RUTA_HORTALIZAS, dtype=str)

print("Forma:", hortalizas.shape)
print("\nColumnas:")
print(hortalizas.columns.tolist())

Forma: (9536, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'produccion_toneladas']


### 7. Producción de frutales (`8xap-gecf`)

In [9]:
URL = f"{BASE_URL}/{ID_FRUTALES}.csv?$limit={LIMITE_COMPLETO}"
RUTA_FRUTALES = "data/frutales.csv"

df_descarga = pd.read_csv(URL, dtype=str)
df_descarga.to_csv(RUTA_FRUTALES, index=False)

frutales = pd.read_csv(RUTA_FRUTALES, dtype=str)

print("Forma:", frutales.shape)
print("\nColumnas:")
print(frutales.columns.tolist())

Forma: (9992, 8)

Columnas:
['anio', 'id_municipio', 'municipio', 'subregion', 'id_cultivo', 'cultivo', 'ciclo', 'produccion_toneladas']


### 8. Temperatura ambiente del aire, IDEAM (`sbwg-7ju4`)

El conjunto tiene **93.573.504 registros**, por lo que no se descarga completo. Se consulta la API con un filtro **en el servidor** por departamento (`VALLE DEL CAUCA`) y se **agrega a promedio mensual por estación y sensor** con SoQL. Se descarga año por año para evitar tiempos de espera, y se guarda en `data/temperatura_valle_mensual.csv`.

Se excluyen en el servidor los valores fuera de -5 a 45 °C (errores de sensor). Las coordenadas y el municipio se obtienen aparte, en un catálogo de estaciones.

In [ ]:
import os
import time
import shutil
import pandas as pd
from urllib.parse import urlencode

COLS_DESC = ("codigoestacion, codigosensor, nombreestacion, departamento, municipio, "
             "zonahidrografica, latitud, longitud, descripcionsensor, unidadmedida")
DTYPES = {"codigoestacion": str, "codigosensor": str}
COLUMNAS_ESPERADAS = 14

os.makedirs("data", exist_ok=True)

# ---------- Función de consulta con reintentos ----------
def consulta_temp(params, intentos=4):
    url = f"{URL_TEMP}?{urlencode(params)}"
    for i in range(intentos):
        try:
            return pd.read_csv(url, dtype=DTYPES)
        except Exception as e:
            print(f"   intento {i+1} falló: {str(e)[:100]}")
            time.sleep(5 * (i + 1))
    raise RuntimeError("La consulta falló tras varios intentos")

# ---------- Borrar archivos viejos (solo si tienen menos columnas) ----------
if os.path.exists(RUTA_TEMP_MENSUAL):
    n_cols = pd.read_csv(RUTA_TEMP_MENSUAL, nrows=1).shape[1]
    if n_cols != COLUMNAS_ESPERADAS:
        print(f"CSV viejo con {n_cols} columnas: se borra y se vuelve a descargar")
        os.remove(RUTA_TEMP_MENSUAL)
        shutil.rmtree(CARPETA_ANUAL, ignore_errors=True)

if os.path.exists(CARPETA_ANUAL):
    for f in os.listdir(CARPETA_ANUAL):
        ruta_f = os.path.join(CARPETA_ANUAL, f)
        if pd.read_csv(ruta_f, nrows=1).shape[1] != COLUMNAS_ESPERADAS:
            shutil.rmtree(CARPETA_ANUAL, ignore_errors=True)
            print("Archivos anuales viejos borrados")
            break

# ---------- Descarga año por año ----------
if os.path.exists(RUTA_TEMP_MENSUAL):
    print(f"{RUTA_TEMP_MENSUAL} ya existe con {COLUMNAS_ESPERADAS} columnas, no se descarga")
else:
    os.makedirs(CARPETA_ANUAL, exist_ok=True)
    partes = []
    for anio in range(ANIO_INI, ANIO_FIN + 1):
        ruta = f"{CARPETA_ANUAL}/temp_{anio}.csv"
        if os.path.exists(ruta):                      # reanuda si se cortó
            df = pd.read_csv(ruta, dtype=DTYPES)
        else:
            where = (f"{WHERE_DEPTO} "
                     f"AND fechaobservacion >= '{anio}-01-01T00:00:00' "
                     f"AND fechaobservacion < '{anio + 1}-01-01T00:00:00' "
                     f"AND valorobservado BETWEEN {T_MIN} AND {T_MAX}")
            df = consulta_temp({
                "$select": (f"{COLS_DESC}, "
                            "date_extract_m(fechaobservacion) as mes, "
                            "avg(valorobservado) as temp_media, "
                            "min(valorobservado) as temp_min, "
                            "max(valorobservado) as temp_max, "
                            "count(*) as n_obs"),
                "$where": where,
                "$group": f"{COLS_DESC}, date_extract_m(fechaobservacion)",
                "$limit": 500000,
            })
            if df.empty:
                # año sin datos: se guarda un CSV vacío con las columnas correctas
                df = pd.DataFrame(columns=COLS_DESC.replace(" ", "").split(",")
                                  + ["mes", "temp_media", "temp_min", "temp_max", "n_obs"])
            df.insert(0, "anio", anio)
            df.to_csv(ruta, index=False)
        print(f"{anio}: {len(df):,} filas")
        if len(df):
            partes.append(df)

    pd.concat(partes, ignore_index=True).to_csv(RUTA_TEMP_MENSUAL, index=False)

# ---------- Lectura y verificación ----------
temp_mensual = pd.read_csv(RUTA_TEMP_MENSUAL, dtype=DTYPES)
print("\nForma:", temp_mensual.shape)
print("Columnas:", temp_mensual.columns.tolist())
assert temp_mensual.shape[1] == COLUMNAS_ESPERADAS, "Faltan columnas: revisa el $select"
print("Estaciones:", temp_mensual["codigoestacion"].nunique())
print("Municipios:", temp_mensual["municipio"].nunique())
print(f"Observaciones crudas resumidas: {temp_mensual['n_obs'].sum():,}")
temp_mensual.head()

2000: 0 filas
2001: 0 filas
2002: 4 filas
2003: 12 filas
2004: 12 filas


# Integración de los conjuntos agrícolas (CSV locales)

En esta sección se trabaja **solo con los CSV guardados en `data/`**, sin volver a consultar la API. Los conjuntos se cruzan por la llave:

**`anio` + `id_municipio` + `id_cultivo` + `ciclo`**

| Grupo | Conjuntos | Qué aporta |
|---|---|---|
| Transitorios | `rend_trans` + `sup_trans` | Rendimiento (t/ha) y hectáreas sembradas |
| Permanentes | `consol_perm` + catálogo de subregiones | Área, producción y rendimiento |
| Solo producción | `raices`, `hortalizas`, `frutales` | Toneladas (sin área ni rendimiento) |

**Aclaraciones:**
- Los archivos `_muestra.csv` se usan los CSV completos.
- `consol_perm` no trae `subregion`: se obtiene de los demás conjuntos por `id_municipio`.
- Como `rend_trans` ya trae t/ha, no se divide producción entre hectáreas.

In [51]:
from urllib.parse import urlencode

VERIFICACION = {
    "raices": (ID_RAICES, raices),
    "rend_trans": (ID_REND_TRANS, rend_trans),
    "sup_trans": (ID_SUP_TRANS, sup_trans),
    "sup_perm": (ID_SUP_PERM, sup_perm),
    "consol_perm": (ID_CONSOL_PERM, consol_perm),
    "hortalizas": (ID_HORTALIZAS, hortalizas),
    "frutales": (ID_FRUTALES, frutales),
}

for nombre, (id_c, df) in VERIFICACION.items():
    q = urlencode({"$select": "count(*)"})
    total = int(pd.read_csv(f"{BASE_URL}/{id_c}.csv?{q}").iloc[0, 0])
    estado = "OK" if len(df) == total else "INCOMPLETO"
    print(f"{nombre:12} CSV: {len(df):>7,} | portal: {total:>7,} | {estado}")

raices       CSV:   1,773 | portal:   1,773 | OK
rend_trans   CSV:  16,748 | portal:  16,748 | OK
sup_trans    CSV:  16,748 | portal:  16,748 | OK
sup_perm     CSV:  14,590 | portal:  14,590 | OK
consol_perm  CSV:  15,282 | portal:  15,282 | OK
hortalizas   CSV:   9,536 | portal:   9,536 | OK
frutales     CSV:   9,992 | portal:   9,992 | OK


## Lectura y estandarización

Se leen los CSV como texto y se estandarizan:

- `astype(int)` para el año.
- `str.zfill(5)` para que `id_municipio` sea código **DIVIPOLA de 5 dígitos**.
- Métodos `str` (`strip`, `title`) para municipio, cultivo y ciclo.
- `pd.to_numeric` para rendimiento, hectáreas y toneladas (los valores inválidos pasan a `NaN`).
- Se renombran las columnas distintas de `consol_perm` (`a_o` → `anio`) y de `rend_trans`, para que todas coincidan.

In [53]:
def leer(nombre):
    return pd.read_csv(f"data/{nombre}.csv", dtype=str)

raices, rend_trans, sup_trans = leer("raices"), leer("rend_trans"), leer("sup_trans")
sup_perm, consol_perm = leer("sup_perm"), leer("consol_perm")
hortalizas, frutales = leer("hortalizas"), leer("frutales")

# Nombres homogéneos
rend_trans = rend_trans.rename(columns={"rendimiento_toneladas_hectareas": "rendimiento_t_ha"})
consol_perm = consol_perm.rename(columns={"a_o": "anio",
                                          "rendimiento_toneladas": "rendimiento_t_ha"})

LLAVE = ["anio", "id_municipio", "id_cultivo", "ciclo"]

def estandarizar(df):
    df = df.copy()
    df["anio"] = df["anio"].astype(int)
    df["id_municipio"] = df["id_municipio"].str.strip().str.zfill(5)
    df["id_cultivo"] = df["id_cultivo"].str.strip()
    df["ciclo"] = df["ciclo"].str.strip().str.title()
    df["municipio"] = df["municipio"].str.strip().str.title()
    df["cultivo"] = df["cultivo"].str.strip().str.title()
    return df

raices, rend_trans, sup_trans = map(estandarizar, (raices, rend_trans, sup_trans))
sup_perm, consol_perm = map(estandarizar, (sup_perm, consol_perm))
hortalizas, frutales = map(estandarizar, (hortalizas, frutales))

# Columnas numéricas
rend_trans["rendimiento_t_ha"] = pd.to_numeric(rend_trans["rendimiento_t_ha"], errors="coerce")
sup_trans["hectareas_sembradas"] = pd.to_numeric(sup_trans["hectareas_sembradas"], errors="coerce")
sup_perm["hectareas_sembradas"] = pd.to_numeric(sup_perm["hectareas_sembradas"], errors="coerce")
for c in ["hectareas_sembradas", "hectareas_cosechadas", "produccion_toneladas", "rendimiento_t_ha"]:
    consol_perm[c] = pd.to_numeric(consol_perm[c], errors="coerce")
for df in (raices, hortalizas, frutales):
    df["produccion_toneladas"] = pd.to_numeric(df["produccion_toneladas"], errors="coerce")

# Tamaño y rango de años
for nombre, df in {"raices": raices, "rend_trans": rend_trans, "sup_trans": sup_trans,
                   "sup_perm": sup_perm, "consol_perm": consol_perm,
                   "hortalizas": hortalizas, "frutales": frutales}.items():
    print(f"{nombre:12} {df.shape}  años {df['anio'].min()}-{df['anio'].max()}")

raices       (1773, 8)  años 2000-2022
rend_trans   (16748, 8)  años 2000-2022
sup_trans    (16748, 8)  años 2000-2022
sup_perm     (14590, 8)  años 2000-2022
consol_perm  (15282, 11)  años 2000-2023
hortalizas   (9536, 8)  años 2000-2022
frutales     (9992, 8)  años 2000-2022


## Revisión de la llave de cruce

Antes de unir, se verifica que la llave sea **única** en cada conjunto. Si hay filas repetidas, el `merge` las duplicaría y alteraría los totales. También se revisan los valores de `ciclo` y `tipo_cultivo`, que definen qué es transitorio y qué es permanente.

In [54]:
for nombre, df in {"rend_trans": rend_trans, "sup_trans": sup_trans,
                   "consol_perm": consol_perm}.items():
    print(f"{nombre}: filas con llave repetida = {df.duplicated(LLAVE).sum()}")

print("\nCiclos en transitorios:", sorted(rend_trans["ciclo"].unique()))
print("Ciclos en permanentes:", sorted(consol_perm["ciclo"].unique()))
print("tipo_cultivo en consol_perm:", consol_perm["tipo_cultivo"].unique())

rend_trans: filas con llave repetida = 456
sup_trans: filas con llave repetida = 456
consol_perm: filas con llave repetida = 18

Ciclos en transitorios: ['Semestre 1', 'Semestre 2']
Ciclos en permanentes: ['Anual']
tipo_cultivo en consol_perm: <StringArray>
[                                                    'Frutales',
                            'Cultivos Tropicales Tradicionales',
                                                   'Hortalizas',
                                          'Raíces y Tubérculos',
                                                       'Flores',
 'Cultivos para Condimentos y Bebidas Medicinales y Aromáticas',
                                          'Raíces y tubérculos',
                                                  'Oleaginosas']
Length: 8, dtype: str


### Diagnóstico de llaves repetidas

Si el paso anterior reporta repetidos, se inspeccionan antes de decidir. Un repetido **exacto** (todas las columnas iguales) se elimina; uno con **valores distintos** puede ser un registro legítimo (por ejemplo, otro ciclo del mismo cultivo) y requiere decidir si se suma o se promedia. La decisión se documenta en el reporte de calidad.

In [55]:
k1 = rend_trans[LLAVE].reset_index(drop=True)
k2 = sup_trans[LLAVE].reset_index(drop=True)

print("Mismo número de filas:", len(rend_trans) == len(sup_trans))
print("Llaves idénticas fila a fila:", k1.equals(k2))
print("Cultivo idéntico fila a fila:",
      rend_trans["cultivo"].reset_index(drop=True).equals(sup_trans["cultivo"].reset_index(drop=True)))
print("Municipio idéntico fila a fila:",
      rend_trans["municipio"].reset_index(drop=True).equals(sup_trans["municipio"].reset_index(drop=True)))

Mismo número de filas: True
Llaves idénticas fila a fila: False
Cultivo idéntico fila a fila: False
Municipio idéntico fila a fila: False


## Catálogo municipio → subregión

`consol_perm` no trae subregión. Se construye un catálogo único `id_municipio → subregion` con los demás conjuntos y se verifica que **cada municipio tenga una sola subregión**; si no, el `merge` duplicaría filas.

In [56]:
catalogo_subregion = (pd.concat([d[["id_municipio", "subregion"]]
                                 for d in (raices, rend_trans, sup_trans, sup_perm,
                                           hortalizas, frutales)])
                      .drop_duplicates())

conflictos = catalogo_subregion.groupby("id_municipio")["subregion"].nunique().gt(1).sum()
print("Municipios con más de una subregión:", conflictos)
print("Subregiones:", sorted(catalogo_subregion["subregion"].dropna().unique()))
print("Municipios en el catálogo:", catalogo_subregion["id_municipio"].nunique())

Municipios con más de una subregión: 0
Subregiones: ['Centro', 'Norte', 'Pacifico', 'Sur']
Municipios en el catálogo: 42


## Merge de transitorios: rendimiento + superficie

Se une `rend_trans` (t/ha) con `sup_trans` (hectáreas sembradas) usando `merge` con `how="left"` e `indicator=True`. Se reporta el **% de registros que cruzaron** y las **causas de los que no** (por cultivo y por año). Estas cifras alimentan la tabla de linaje de la Parte 2.

Se usa `validate="one_to_one"`: si la llave se repitiera, el código falla a propósito en lugar de duplicar filas en silencio.

In [57]:
rend_trans["orden"] = rend_trans.groupby(LLAVE).cumcount()
sup_trans["orden"] = sup_trans.groupby(LLAVE).cumcount()

transitorios = rend_trans.merge(
    sup_trans[LLAVE + ["orden", "hectareas_sembradas"]],
    on=LLAVE + ["orden"], how="left", indicator=True, validate="one_to_one",
)

pct_trans = transitorios["_merge"].eq("both").mean() * 100
print(f"Registros de rendimiento con superficie: {pct_trans:.1f}%")
print(f"Filas: {len(rend_trans):,} -> {len(transitorios):,} (deben ser iguales)")

sin_cruce = transitorios[transitorios["_merge"] == "left_only"]
print("\nSin cruce por cultivo (top 10):")
print(sin_cruce["cultivo"].value_counts().head(10))
print("\nSin cruce por año:")
print(sin_cruce["anio"].value_counts().sort_index())

transitorios = transitorios.drop(columns=["_merge", "orden"]).assign(grupo="Transitorio")

Registros de rendimiento con superficie: 100.0%
Filas: 16,748 -> 16,748 (deben ser iguales)

Sin cruce por cultivo (top 10):
Series([], Name: count, dtype: int64)

Sin cruce por año:
Series([], Name: count, dtype: int64)


## Merge de permanentes: consolidado + subregión

`consol_perm` ya trae hectáreas, producción y rendimiento. Solo se le agrega la **subregión** desde el catálogo (`merge` por `id_municipio`, relación muchos a uno). Se reportan los municipios que quedan sin subregión.

In [58]:
permanentes = consol_perm.merge(catalogo_subregion, on="id_municipio",
                                how="left", indicator=True, validate="many_to_one")

pct_perm = permanentes["_merge"].eq("both").mean() * 100
print(f"Permanentes con subregión asignada: {pct_perm:.1f}%")
print("Municipios sin subregión:",
      permanentes.loc[permanentes["_merge"] == "left_only", "municipio"].unique())

permanentes = permanentes.drop(columns="_merge").assign(grupo="Permanente")

Permanentes con subregión asignada: 100.0%
Municipios sin subregión: <StringArray>
[]
Length: 0, dtype: str


## Tabla unificada de rendimiento

Se apilan transitorios y permanentes (`concat`) en una sola tabla con las mismas columnas y se resume el rendimiento por grupo con `describe`. Después se calcula el **rendimiento promedio ponderado por hectáreas sembradas** por grupo, subregión y año: un promedio simple trataría igual a un municipio de 5 ha y a uno de 5.000 ha.

In [59]:
COLS = ["anio", "id_municipio", "municipio", "subregion", "id_cultivo",
        "cultivo", "ciclo", "grupo", "rendimiento_t_ha", "hectareas_sembradas"]

rendimiento = pd.concat([transitorios[COLS], permanentes[COLS]], ignore_index=True)
print(rendimiento.shape)
display(rendimiento.groupby("grupo")["rendimiento_t_ha"].describe())

def ponderado(g):
    g = g.dropna(subset=["rendimiento_t_ha", "hectareas_sembradas"])
    suma = g["hectareas_sembradas"].sum()
    return (g["rendimiento_t_ha"] * g["hectareas_sembradas"]).sum() / suma if suma > 0 else float("nan")

serie = (rendimiento.groupby(["grupo", "subregion", "anio"])
         .apply(ponderado, include_groups=False)
         .rename("rend_ponderado").reset_index())
serie.head()

(32030, 10)


,count,mean,std,min,25%,50%,75%,max
grupo,,,,,,,,
Permanente,15282.0,19.593231,26.832170,0.0,6.0,10.0,18.8975,190.0
Transitorio,16748.0,10.258067,11.314402,0.0,2.4,6.8,15.0000,240.0


,grupo,subregion,anio,rend_ponderado
0,Permanente,Centro,2000,60.002637
1,Permanente,Centro,2001,51.982172
2,Permanente,Centro,2002,56.455368
3,Permanente,Centro,2003,54.049337
4,Permanente,Centro,2004,57.887955


## Revisión de ceros y valores extremos

El resumen de la tabla unificada muestra rendimientos mínimos de 0 y máximos de 190 t/ha (permanentes) y 240 t/ha (transitorios). Se revisa:

- **Ceros:** pueden ser área sembrada que no se cosechó o datos faltantes. Se cuantifican antes de decidir si se excluyen del promedio.
- **Extremos:** se comparan con la distribución de **cada cultivo**, porque un valor alto puede ser normal para un cultivo y un error para otro.

In [60]:
# Ceros por grupo
ceros = rendimiento.groupby("grupo")["rendimiento_t_ha"].apply(lambda s: (s == 0).sum())
nulos = rendimiento.groupby("grupo")["rendimiento_t_ha"].apply(lambda s: s.isna().sum())
print("Rendimiento = 0:\n", ceros, "\n\nRendimiento nulo:\n", nulos)

# Cultivos con más rendimiento (percentil 99 y máximo)
por_cultivo = (rendimiento[rendimiento["rendimiento_t_ha"] > 0]
               .groupby(["grupo", "cultivo"])["rendimiento_t_ha"]
               .agg(n="size", mediana="median", p99=lambda s: s.quantile(0.99), maximo="max")
               .assign(razon_max_mediana=lambda d: d["maximo"] / d["mediana"])
               .sort_values("razon_max_mediana", ascending=False))
display(por_cultivo.head(15))

# Qué cultivos dominan las hectáreas
ha = (rendimiento.groupby(["grupo", "cultivo"])["hectareas_sembradas"].sum()
      .sort_values(ascending=False))
print("\nTop 10 cultivos por hectáreas acumuladas:")
display((ha / ha.sum() * 100).round(1).head(10).rename("% de hectáreas"))

Rendimiento = 0:
 grupo
Permanente     302
Transitorio    263
Name: rendimiento_t_ha, dtype: int64 

Rendimiento nulo:
 grupo
Permanente     0
Transitorio    0
Name: rendimiento_t_ha, dtype: int64


n  mediana      p99  maximo  \
grupo       cultivo                                              
Permanente  Cacao                560     0.60   2.0000   45.00   
Transitorio Fríjol Plana         175     1.20   6.1574   42.00   
            Pimentón            1080    12.00  71.2600  173.00   
            Repollo              560    18.00  55.3300  240.00   
Permanente  Aromáticas            23     5.00  42.7756   52.02   
Transitorio Maíz Ladera         1161     2.00   5.0000   19.43   
            Aromáticas           215     5.50  18.0000   52.00   
            Frijol Tradicional   496     1.15   4.0030   10.00   
            Fríjol Ladera       1241     0.82   3.1200    7.00   
            Cimarrón             147     6.50  20.5400   54.00   
            Habichuela          1159     8.00  20.0000   65.00   
            Arveja               595     2.50   8.0000   20.00   
Permanente  Yuca                 767    14.00  30.0000  108.00   
Transitorio Tomate              1658    20.00  61.0750  148.00   
            Maíz Plana           852     5.50   9.7488   40.00   

                                razon_max_mediana  
grupo       cultivo                                
Permanente  Cacao                       75.000000  
Transitorio Fríjol Plana                35.000000  
            Pimentón                    14.416667  
            Repollo                     13.333333  
Permanente  Aromáticas                  10.404000  
Transitorio Maíz Ladera                  9.715000  
            Aromáticas                   9.454545  
            Frijol Tradicional           8.695652  
            Fríjol Ladera                8.536585  
            Cimarrón                     8.307692  
            Habichuela                   8.125000  
            Arveja                       8.000000  
Permanente  Yuca                         7.714286  
Transitorio Tomate                       7.400000  
            Maíz Plana                   7.272727


Top 10 cultivos por hectáreas acumuladas:


grupo        cultivo         
Permanente   Caña De Azúcar      51.9
             Café                18.7
             Plátano              6.3
Transitorio  Maíz Plana           3.8
Permanente   Caña Panelera        1.7
             Banano               1.6
Transitorio  Maíz Tecnificado     1.4
Permanente   Cítricos             1.3
Transitorio  Arroz                1.2
             Soya                 1.1
Name: % de hectáreas, dtype: float64

## Ceros por año y regla de valores extremos

**Ceros.** Hay registros con rendimiento 0 . Un cero con hectáreas sembradas puede ser pérdida de cosecha, que es relevante para la hipótesis climática. Se revisa si se concentran en ciertos años antes de excluirlos.

**Extremos.** Los máximos de cacao (45 t/ha, mediana 0,6), fríjol plana (42 t/ha, mediana 1,2) y repollo (240 t/ha) son errores de digitación evidentes. Un recorte al percentil 99 elimina siempre el 1 % de cada cultivo y no actúa en cultivos con pocos registros. Se usa en cambio una regla relativa: se marca como atípico todo rendimiento **mayor a 5 veces la mediana de su cultivo**. Se lista cuántos registros excluye por cultivo para revisar si la regla es razonable.

In [61]:
# 1. ¿Los ceros se concentran en ciertos años?
r0 = rendimiento.assign(es_cero=rendimiento["rendimiento_t_ha"].eq(0))
ceros_anio = (r0.groupby(["grupo", "anio"])["es_cero"].agg(ceros="sum", n="size")
              .assign(pct=lambda d: (d["ceros"] / d["n"] * 100).round(1)))
display(ceros_anio["pct"].unstack("grupo"))

# ¿Qué hectáreas representan los ceros?
print("Hectáreas sembradas en registros con rendimiento 0:")
display(r0[r0["es_cero"]].groupby("grupo")["hectareas_sembradas"].agg(["count", "sum", "median"]))

# 2. Regla relativa: más de 5 veces la mediana del cultivo
FACTOR = 5
pos = rendimiento[rendimiento["rendimiento_t_ha"] > 0].copy()
pos["mediana"] = pos.groupby(["grupo", "id_cultivo"])["rendimiento_t_ha"].transform("median")
pos["atipico"] = pos["rendimiento_t_ha"] > FACTOR * pos["mediana"]

print(f"\nRegistros atípicos (> {FACTOR}x mediana): {pos['atipico'].sum():,} "
      f"de {len(pos):,} ({pos['atipico'].mean():.2%})")
display(pos[pos["atipico"]].groupby(["grupo", "cultivo"])
        .agg(excluidos=("atipico", "size"),
             hectareas=("hectareas_sembradas", "sum"),
             max_t_ha=("rendimiento_t_ha", "max"))
        .sort_values("excluidos", ascending=False).head(15))

grupo,Permanente,Transitorio
anio,,
2000,2.4,0.8
2001,1.6,0.8
2002,0.8,0.1
2003,0.0,0.0
2004,1.9,0.6
2005,0.6,0.1
2006,0.9,0.5
2007,2.0,0.5
2008,3.4,3.7


Hectáreas sembradas en registros con rendimiento 0:


,count,sum,median
grupo,,,
Permanente,302,18402.110,5.0
Transitorio,263,1467.875,0.0



Registros atípicos (> 5x mediana): 62 de 31,465 (0.20%)


excluidos  hectareas  max_t_ha
grupo       cultivo                                           
Transitorio Pimentón                   15      330.0    173.00
Permanente  Papaya                      6      323.0    190.00
            Cacao                       4       98.5     45.00
Transitorio Maíz Ladera                 4      115.5     19.43
Permanente  Flores Y Follajes           3        3.3     30.00
            Sábila                      2        3.5    120.00
            Guayaba                     2        3.6     56.11
Transitorio Habichuela                  2       17.0     65.00
Permanente  Yuca                        2        8.0    108.00
            Café                        2     1844.0      5.00
Transitorio Fríjol Plana                2       10.0     42.00
            Frijol Tradicional          2       26.0     10.00
            Maíz Plana                  2      340.0     40.00
Permanente  Guanábana                   1       10.6     70.00
            Arracacha                   1        8.0     53.12